# dlt-doctor: Production Optimization Engine for Databricks DLT & PySpark

This interactive walkthrough demonstrates how `dlt-doctor` catches Catalyst plan bloat, enforces zero-loss quarantine streams, deduplicates streaming CDC micro-batches, and calculates partition-pruned Delta restatements.

- **Documentation**: [https://github.com/MsMansiDhruv/dlt-doctor](https://github.com/MsMansiDhruv/dlt-doctor)
- **PyPI**: `pip install dlt-doctor`

In [ ]:
# 1. Install or upgrade dlt-doctor on Databricks
%pip install --upgrade dlt-doctor

In [ ]:
# 2. Load the interactive Databricks cell magic extension
%load_ext dlt_doctor

## Act 1: Interactive Catalyst AST Auditing

Prefix any notebook cell with `%%dlt_doctor` to statically inspect the PySpark Catalyst plan before executing cluster compute.

In [ ]:
%%dlt_doctor Orders Staging View
from pyspark.sql import functions as F

# Anti-Pattern: Chained .withColumn() calls adding recursive Project nodes
df_orders = (
    spark.readStream.table("bronze_orders")
    .withColumn("quantity_clean", F.expr("try_cast(quantity AS INT)"))
    .withColumn("unit_price_clean", F.expr("try_cast(unit_price AS DECIMAL(18,2))"))
    .withColumn("total_amount_clean", F.expr("try_cast(total_amount AS DECIMAL(18,2))"))
    .withColumn("order_timestamp_clean", F.expr("try_cast(order_timestamp AS TIMESTAMP)"))
)

### Refactored: Consolidated `withColumns`

Consolidating projections into a single Spark 3.3+ `withColumns({...})` call eliminates Catalyst plan compilation overhead.

In [ ]:
%%dlt_doctor Orders Staging Clean
from pyspark.sql import functions as F

# Refactored: Single Project node in Catalyst logical plan
df_orders_clean = spark.readStream.table("bronze_orders").withColumns({
    "quantity_clean": F.expr("try_cast(quantity AS INT)"),
    "unit_price_clean": F.expr("try_cast(unit_price AS DECIMAL(18,2))"),
    "total_amount_clean": F.expr("try_cast(total_amount AS DECIMAL(18,2))"),
    "order_timestamp_clean": F.expr("try_cast(order_timestamp AS TIMESTAMP)"),
})

## Act 2: Dual-Routing Quarantine Tables (Zero Silent Row Loss)

Native DLT `expect_or_drop` silently purges invalid records. `ExpectationDoctor` generates resilient dual-routing streaming flows: valid records flow into Silver, while non-compliant records land in a quarantine table for audit and remediation.

In [ ]:
import dlt
from pyspark.sql import functions as F
from dlt_doctor import ExpectationDoctor, ExpectationRule

doctor = ExpectationDoctor()
rules = [
    ExpectationRule("valid_id", "order_id IS NOT NULL"),
    ExpectationRule("positive_price", "unit_price > 0"),
    ExpectationRule("valid_email", "email LIKE '%@%.%'"),
]
routing = doctor.build_quarantine_routing("silver_orders", rules)

@dlt.table(name=routing.target_clean_table)
def silver_orders():
    return dlt.read_stream("bronze_orders").filter(routing.valid_filter_sql)

@dlt.table(name=routing.target_quarantine_table)
def silver_orders_quarantine():
    return (
        dlt.read_stream("bronze_orders")
        .filter(routing.invalid_filter_sql)
        .withColumn("_quarantined_at", F.current_timestamp())
    )

## Act 3: Late Data Restatement (98.1% I/O Scan Reduction)

Instead of running a full-table historical reload when late records arrive, `RestatementDoctor` calculates the exact partition blast radius and outputs a surgical partition-pruned `MERGE` statement.

In [ ]:
from dlt_doctor import RestatementDoctor

reconciler = RestatementDoctor()
scope = reconciler.calculate_blast_radius(
    target_table="main.gold.fact_orders",
    late_date_iso="2026-10-25",
    window_size_days=7,
    partition_col="order_date",
    match_key="order_id",
)

print(f"Target Table: {scope.target_table}")
print(f"Blast Radius Window: {scope.blast_radius_start} to {scope.blast_radius_end}")
print("\nSurgical Delta MERGE SQL:")
print(scope.delta_merge_sql)

## Act 4: Liquid Clustering Advisor & VACUUM Reader Guard

Select optimal `CLUSTER BY` keys and audit Delta Lake retention against active streaming reader states.

In [ ]:
from dlt_doctor import ClusterDoctor, ColumnProfile, VacuumDoctor

# 1. Liquid Clustering Advisor
clustering_rec = ClusterDoctor().advise_clustering(
    table_name="main.gold.fact_orders",
    columns=[
        ColumnProfile("customer_id", "string", distinct_count=25000, filter_frequency_score=0.9),
        ColumnProfile("order_date", "date", distinct_count=1800, filter_frequency_score=0.95),
        ColumnProfile("order_uuid", "string", distinct_count=9900000),  # Rejected: near-unique
    ],
    total_rows=10000000,
)
print("Liquid Clustering Recommendation:")
print(clustering_rec.alter_cluster_ddl)

# 2. Safe Delta VACUUM Retention Guard
vac_report = VacuumDoctor().evaluate_vacuum_safety(
    table_name="main.silver.orders",
    requested_retention_hours=24,
    active_streaming_readers=2,
    cdf_enabled=True,
)
print(f"\nVacuum Safety Status: {vac_report.safety_level.value}")
print(f"Reason: {vac_report.reason}")
print(f"Recommended SQL: {vac_report.recommended_vacuum_sql}")